# Built-In Tools

### Function Calling
Already covered in details in another notebook.

### File Search
This is OpenAI's answer to RAG. Essentially I can upload a bunch of files to their server, create a vector db instance out of these files. And then when making a completion call, specify the `file_search` tool along with a handle to the vector db. Will not bothering trying it out. Have tried out RAG a bunch of times by now and if it comes to that, I'll mostly not use OpenAI's RAG for any of my projects.

### Web Search
Searches the web and builds context to give to LLM. With the Chat API I have to use the "gpt-4o-search-preview" model. Optionally, I can specify web search options like user location with it. With the Responses API, this model will not work. I need to use the usual 4.1 or 4.o models but I have to specify the `web_search_preview` tool.

### Computer Use
Given a high level description of a task and the screenshot of the computer, the model outputs a user action (e.g., clicking the mouse button at a specific cooridnate, scrolling, etc.). The user can then take the action and call the model again with a screenshot of the computer and so on. As of this writing (4/17/2025) this works mostly with browser runners like Playwright or Selenium. I am not demoing it here because I need to think of a good enough scenario for this.



In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from utils import LLM

In [2]:
load_dotenv()

True

In [3]:
client = OpenAI()

In [4]:
completion = client.chat.completions.create(
    model=LLM.PRE,
    messages=[
        {"role": "user", "content": "What was a positive news story from today?"}
    ],
)
completion

ChatCompletion(id='chatcmpl-BNESnk6FW249oHLqbXAAf2oiSdMpo', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I don’t have access to real-time news updates, but I can share a positive story that’s recently been reported.\n\n**Example of a recent positive news story:**\n\n*A new study has found that global rates of extreme poverty have declined significantly over the past decade, according to the World Bank. Thanks to improved access to education, healthcare, and renewable energy, millions of people have been lifted out of poverty, particularly in regions of Asia and Africa. Experts say this shows the power of international cooperation and targeted development projects in improving lives.*\n\nIf you’re looking for a positive story from today specifically, I suggest checking the “Good News” sections of reputable news outlets such as BBC, NPR, or The Guardian, which regularly feature uplifting and inspiring stories. If you have a particul

```python
ChatCompletion(
    id='chatcmpl-BNESnk6FW249oHLqbXAAf2oiSdMpo', 
    choices=[
        Choice(
            finish_reason='stop', 
            index=0, 
            logprobs=None, 
            message=ChatCompletionMessage(
                content='I don’t have access to real-time news updates, but I can...', 
                refusal=None, 
                role='assistant', 
                annotations=[], 
                audio=None, 
                function_call=None, 
                tool_calls=None
            )
        )
    ], 
    created=1744876841, 
    model='gpt-4.1-2025-04-14', 
    object='chat.completion', 
    service_tier='default', 
    system_fingerprint='fp_a1102cf978', 
    usage=CompletionUsage(
        completion_tokens=183, 
        prompt_tokens=16, 
        total_tokens=199, 
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0, 
            audio_tokens=0, 
            reasoning_tokens=0, 
            rejected_prediction_tokens=0
        ), 
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    )
)
```

In [5]:
print(completion.choices[0].message.content)

I don’t have access to real-time news updates, but I can share a positive story that’s recently been reported.

**Example of a recent positive news story:**

*A new study has found that global rates of extreme poverty have declined significantly over the past decade, according to the World Bank. Thanks to improved access to education, healthcare, and renewable energy, millions of people have been lifted out of poverty, particularly in regions of Asia and Africa. Experts say this shows the power of international cooperation and targeted development projects in improving lives.*

If you’re looking for a positive story from today specifically, I suggest checking the “Good News” sections of reputable news outlets such as BBC, NPR, or The Guardian, which regularly feature uplifting and inspiring stories. If you have a particular area of interest (science, environment, health, etc.), let me know and I can share a recent positive development in that field!


In [6]:
resp = client.responses.create(
    model=LLM.PRE,
    tools=[{"type": "web_search_preview"}],
    input="What was a positive news story from today?",
)

In [7]:
print(resp.output_text)

As of April 17, 2025, one notable positive news story is the Democratic Republic of Congo's (DRC) commitment to establish the world's largest protected tropical reserve, the Couloir Vert, encompassing an area approximately the size of France. This initiative aims to conserve vast tracts of the Congo Basin, including forests and peatlands, which serve as the largest tropical carbon sink globally and are home to diverse species such as the eastern lowland gorillas. While the project has been welcomed for its potential environmental benefits, concerns have been raised about the need for more consultation with local communities and stakeholders. ([positive.news](https://www.positive.news/society/good-news-stories-from-week-04-of-2025/?utm_source=openai)) 


In [8]:
completion = client.chat.completions.create(
    model=LLM.WEB_SEARCH,
    messages=[
        {"role": "user", "content": "What was a positive news story from today?"}
    ],
)

In [9]:
print(completion.choices[0].message.content)

As of April 17, 2025, one notable positive news story is the Democratic Republic of Congo's (DRC) commitment to establish the world's largest protected tropical reserve, known as the Couloir Vert or 'Green Corridor.' This expansive reserve, approximately the size of France, aims to conserve vast areas of the Congo Basin, including critical forests and peatlands. These ecosystems are vital as they serve as the largest tropical carbon sink globally and are home to diverse species, such as the eastern lowland gorillas. The initiative underscores the DRC's dedication to addressing climate change and preserving biodiversity. ([positive.news](https://www.positive.news/society/good-news-stories-from-week-04-of-2025/?utm_source=openai)) 


In [10]:
resp = client.responses.create(
    model=LLM.PRE,
    # tools=[{"type": "web_search_preview"}],
    input="What was a positive news story from today?",
)
print(resp.output_text)

Certainly! Here’s a positive news story from today (June 29, 2024):

**Major Sea Turtle Nesting Boom Observed on Southeastern U.S. Beaches**

Marine biologists have reported a record boom in sea turtle nesting along the southeastern U.S. coastline this summer. Conservationists monitoring beaches in Florida, Georgia, and the Carolinas have documented tens of thousands of loggerhead turtle nests, with early estimates suggesting the highest numbers in more than a decade.

Experts attribute the population surge to years of dedicated conservation efforts, including stricter protections for nesting areas, reduction in beachfront lighting, and community engagement initiatives. The increase is being hailed as a sign that sustained environmental stewardship can help endangered species recover and thrive.

***
If you’re interested in other areas—like science, technology, or health—let me know, and I can share more uplifting news!


In [12]:
from openai import BadRequestError


try:
    resp = client.responses.create(
        model=LLM.WEB_SEARCH,
        # tools=[{"type": "web_search_preview"}],
        input="What was a positive news story from today?",
    )
    print(resp.output_text)
except BadRequestError as err:
    print(err)

Error code: 400 - {'error': {'message': "The requested model 'gpt-4o-search-preview' is not supported with the Responses API.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_found'}}


In [14]:
resp = client.responses.create(
    model=LLM.PRE_FAST,
    tools=[{"type": "web_search_preview"}],
    input="Check Avilay Parekh's Linked In profile and tell me if he is a good fit for a Machine Learning Engineer position in my team.",
)
print(resp.output_text)

Based on the available information, Avilay Parekh has a substantial background in machine learning and software engineering. He was among the original creators of Microsoft's Windows Azure platform, contributed to machine learning applications in IoT startups, worked as an engineer at Amazon, and developed 3D simulations on the Unity platform for AI model training. ([ai-podcast.com](https://www.ai-podcast.com/episodes/interview-with-avilay-parekh?utm_source=openai))

While these experiences demonstrate his expertise in machine learning and related fields, determining if he is a good fit for your team would require more specific information about your team's needs, the particular responsibilities of the Machine Learning Engineer position, and how his skills and experiences align with those requirements. 
